In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
def preprocess_data():
    train_data = pd.read_excel("traindata.xlsx")   
    hotCoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    hot1 = hotCoder.fit_transform(train_data[["I"]])
    hot1 = pd.DataFrame(hot1, columns=["EE","SSD","MC"])
    train_data = train_data.drop("I", axis=1)
    train_data = pd.concat([train_data, hot1], axis=1)
    # print(train_data)
    X_train = train_data.iloc[:, [0, 1, 2, 3, -1, -2, -3]].values
    y_train = train_data.iloc[:, [-4]].values

    val_data = pd.read_excel("valdata.xlsx")
    hot1_val = hotCoder.transform(val_data[["I"]])
    hot1_val = pd.DataFrame(hot1_val, columns=["EE","SSD","MC"])
    val_data = val_data.drop("I", axis=1)
    val_data = pd.concat([val_data, hot1_val], axis=1)
    X_val = val_data.iloc[:, [0, 1, 2, 3, -1, -2, -3]].values
    y_val = val_data.iloc[:, [-4]].values

    test_data = pd.read_excel("testdata.xlsx")
    hot1_test = hotCoder.transform(test_data[["I"]])
    hot1_test = pd.DataFrame(hot1_test, columns=["EE","SSD","MC"])
    test_data = test_data.drop("I", axis=1)
    test_data = pd.concat([test_data, hot1_test], axis=1)   
    X_test = test_data.iloc[:, [0, 1, 2, 3, -1, -2, -3]].values
    y_test = test_data.iloc[:, [-4]].values
    
    return X_train, y_train, X_val, y_val, X_test, y_test
X_train, y_train, X_val, y_val, X_test, y_test = preprocess_data()
X_train_np = X_train.astype(np.float32)
y_train_np = np.log(y_train.astype(np.float32))
X_val_np = X_val.astype(np.float32)
y_val_np = np.log(y_val.astype(np.float32))
X_test_np = X_test.astype(np.float32)
y_test_np = np.log(y_test.astype(np.float32))
X_train = torch.tensor(X_train_np)
y_train = torch.tensor(y_train_np)
X_val = torch.tensor(X_val_np)
y_val = torch.tensor(y_val_np)
print(X_train.shape, X_val.shape, X_test.shape)
print(y_test)

In [ ]:
class EnhancedGAMWithPairwiseInteraction(nn.Module):
    def __init__(self, input_dim, hidden_units1=32, hidden_units2=16):
        super().__init__()
        self.input_dim = input_dim
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.global_bias = nn.Parameter(torch.tensor([0.0], device=self.device))
        
        self.main_effect_nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, hidden_units1),
                nn.ReLU(),
                nn.Linear(hidden_units1, hidden_units2),
                nn.ReLU(),
                nn.Linear(hidden_units2, 1),
            ).to(self.device)
            for _ in range(input_dim)
        ])
        
        self.pair_interact_nets = nn.ModuleDict() 
        self.interact_attention = nn.ParameterDict() 
        for i in range(input_dim):
            for j in range(i+1, input_dim):
                self.pair_interact_nets[f"net_{i}_{j}"] = nn.Sequential(
                    nn.Linear(2, hidden_units1//2),
                    nn.ReLU(),
                    nn.Linear(hidden_units1//2, hidden_units2//2),
                    nn.ReLU(),
                    nn.Linear(hidden_units2//2, 1),
                ).to(self.device)
                self.interact_attention[f"alpha_{i}_{j}"] = nn.Parameter(torch.tensor([0.1], device=self.device))
    
    def forward(self, x):
        x = x.to(self.device)
        main_effects = torch.zeros(x.shape[0], 1, device=self.device)
        for i in range(self.input_dim):
            xi = x[:, i].unsqueeze(1)
            main_effects += self.main_effect_nets[i](xi)

        interact_effects = torch.zeros(x.shape[0], 1, device=self.device)
        for i in range(self.input_dim):
            for j in range(i+1, self.input_dim):
                xi_xj = torch.cat([x[:, i].unsqueeze(1), x[:, j].unsqueeze(1)], dim=1)
                pair_net = self.pair_interact_nets[f"net_{i}_{j}"]
                alpha = self.interact_attention[f"alpha_{i}_{j}"]
                interact_effects += alpha * pair_net(xi_xj)
        
        total_output = self.global_bias + main_effects + interact_effects
        return total_output
    
    def get_main_effect(self, feature_idx, x_range):      
        x_range_reshaped = x_range.reshape(-1, 1).astype(np.float32)
        x_tensor = torch.tensor(x_range_reshaped, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            main_effect = self.main_effect_nets[feature_idx](x_tensor).cpu().numpy()
        
        return main_effect

In [ ]:
input_dim = 7 
model = EnhancedGAMWithPairwiseInteraction(input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 2000
train_losses = []
Val_losses = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred_train = model(X_train)
    loss_train = criterion(y_pred_train, y_train.to(model.device))
    loss_train.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        y_pred_Val = model(X_val)
        loss_Val = criterion(y_pred_Val, y_val.to(model.device))
    
    train_losses.append(loss_train.item())
    Val_losses.append(loss_Val.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {loss_train.item():.6f}, Val Loss: {loss_Val.item():.6f}")

In [ ]:
model_params_save_path = "trained_model.pth"
torch.save(model.state_dict(), model_params_save_path)

In [ ]:
input_dim = 7
hidden_units1 = 32
hidden_units2 = 16
model = EnhancedGAMWithPairwiseInteraction(
    input_dim=input_dim,
    hidden_units1=hidden_units1,
    hidden_units2=hidden_units2
)

model_params_save_path = "trained_model.pth"
model.load_state_dict(torch.load(model_params_save_path))

model.eval()

In [ ]:
excel_save_path = "contribution.xlsx"
n_samples = 100
param_indices = [0, 1, 2, 3]
sheet_names = [f"Param_{idx}" for idx in param_indices]

with pd.ExcelWriter(excel_save_path, engine='openpyxl') as writer:
    for param_idx, sheet_name in zip(param_indices, sheet_names):
        x_param_range = np.linspace(
            X_train_np[:, param_idx].min(),
            X_train_np[:, param_idx].max(),
            n_samples
        )
        
        main_effect_param = model.get_main_effect(param_idx, x_param_range)
        main_effect_param_1d = main_effect_param.flatten() 
        
        pred_df = pd.DataFrame({
            f"Param_{param_idx}_Value": x_param_range,
            f"Param_{param_idx}_MainEffect": main_effect_param_1d
        })
        
        pred_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
            header=False
        )

In [ ]:
model.eval()
X_train = torch.tensor(X_train_np)
y_train = torch.tensor(y_train_np)
with torch.no_grad():
    y_pred_train_tensor = model(X_train)
    y_pred_train_np = y_pred_train_tensor.cpu().numpy().flatten()
    y_train_np_flatten = y_train_np.flatten()
    print(y_train_np_flatten)
    ss_res = np.sum((y_train_np_flatten - y_pred_train_np) ** 2)
    ss_tot = np.sum((y_train_np_flatten - np.mean(y_train_np_flatten)) ** 2)
    r2_manual = 1 - (ss_res / ss_tot)

print("\n" + "="*50)
print(f"{r2_manual:.6f}")
print("="*50)

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_val_tensor = model(X_val)
    y_pred_val_np = y_pred_val_tensor.cpu().numpy().flatten()
    y_val_np_flatten = y_val_np.flatten()
    
    ss_res = np.sum((y_val_np_flatten - y_pred_val_np) ** 2)
    ss_tot = np.sum((y_val_np_flatten - np.mean(y_val_np_flatten)) ** 2)
    r2_manual = 1 - (ss_res / ss_tot)

print("\n" + "="*50)
print(f"{r2_manual:.6f}")
print("="*50)

In [ ]:
model.eval()
X_test = torch.tensor(X_test_np)
y_test = torch.tensor(y_test_np)
with torch.no_grad():
    y_pred_test_tensor = model(X_test)
    y_pred_test_np = y_pred_test_tensor.cpu().numpy().flatten()
    y_test_np_flatten = y_test_np.flatten()
    
    ss_res = np.sum((y_test_np_flatten - y_pred_test_np) ** 2)
    ss_tot = np.sum((y_test_np_flatten - np.mean(y_test_np_flatten)) ** 2)
    r2_manual = 1 - (ss_res / ss_tot)

print("\n" + "="*50)
print(f"{r2_manual:.6f}")
print("="*50)